# Lesson 6.4: z-Tests for Means and Proportions

Run a z-test for a mean (the 50 bolts) and a z-test for a proportion (184 of 200 customers satisfied), by hand and with statsmodels.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A factory line has always made 10 mm bolts with a known spread of 0.5 mm. An engineer samples 50 bolts and finds a mean of 10.14 mm: has the process drifted? The same team also checks a company's claim of 95% customer satisfaction, when a survey of 200 customers finds 184 satisfied (92%). Is the true satisfaction rate really lower than claimed?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: A z-test for a mean, by hand

H0: the true mean bolt diameter is 10 mm. H1: it is not 10 mm (it has drifted). Sigma = 0.5 mm is known, 50 bolts average 10.14 mm. The z statistic is the gap divided by the standard error, and the two-sided p-value comes from the normal curve.

In [1]:
import numpy as np  # Load NumPy for the square root, under the short name np.
from scipy import stats  # Load SciPy's statistics tools; stats.norm gives the normal curve for the p-value.

mu0, sigma, n, xbar = 10, 0.5, 50, 10.14  # Store the claim (10 mm), the known sigma (0.5), the sample size (50) and the sample mean (10.14) in one line.

se = sigma / np.sqrt(n)  # Standard error: how much a mean of 50 bolts wobbles. It is sigma divided by the square root of n.
z = (xbar - mu0) / se  # The z statistic: how many standard errors our mean is from the claim.
p = 2 * stats.norm.sf(abs(z))  # Two-sided p-value: the area beyond |z| in the upper tail, doubled to include the lower tail ('drifted either way').
ci = (xbar - 1.96 * se, xbar + 1.96 * se)  # A 95% confidence interval: the mean plus or minus 1.96 standard errors. It shows the plausible values of the true mean.

print(f'z = {z:.3f}, p = {p:.4f}')  # Print z and p, rounded, so we can compare them with the lesson's hand calculation.
print(f'95% CI: ({ci[0]:.3f}, {ci[1]:.3f})')  # Print the interval. If it excludes 10, that agrees with rejecting H0.

z = 1.980, p = 0.0477
95% CI: (10.001, 10.279)


**What this means.** z = 1.980 and p = 0.0477. If the process were still centred on 10 mm, a sample mean this far away in either direction would turn up about 4.8% of the time. That is just below 0.05, so we reject H0, but it is borderline. The interval 10.001 to 10.279 only just excludes 10, which tells the same story.

## Step 2: The same test with ztest()

statsmodels needs raw data. We build 50 pretend bolt readings with mean 10.14 and SD 0.5, so the result matches the hand calculation. Note that ztest() estimates the SD from the data rather than using a known sigma.

In [2]:
from statsmodels.stats.weightstats import ztest  # Load ztest from statsmodels, which runs the z-test on raw data instead of by hand.

rng = np.random.default_rng(42)  # Create a seeded random generator so the pretend bolts are the same every time.
x = rng.normal(size=50)  # Draw 50 random numbers from a standard normal curve as raw material for the pretend bolts.
bolts = 10.14 + 0.5 * (x - x.mean()) / x.std(ddof=1)   # mean 10.14, SD 0.5  # Rescale them so the 50 values have exactly mean 10.14 and SD 0.5, matching the lesson. We centre, divide by their SD, then stretch.

z_stat, p_val = ztest(bolts, value=10, alternative='two-sided')  # Run the z-test on the readings against 10, two-sided. ztest returns the z statistic and the p-value.
print(f'mean = {bolts.mean():.2f}, sd = {bolts.std(ddof=1):.2f}')  # Print the mean and SD of the pretend bolts, to confirm they match the lesson's numbers.
print(f'z = {z_stat:.3f}, p = {p_val:.4f}')  # Print z and p so we can check that they match the by-hand answer.

mean = 10.14, sd = 0.50
z = 1.980, p = 0.0477


**What this means.** The pretend bolts have mean 10.14 and SD 0.50, and ztest gives z = 1.980 and p = 0.0477, the same as the hand calculation. One detail: ztest estimates the SD from the data (it happens to equal 0.5 here), whereas in a real z-test the known sigma is supplied by the problem.

## Step 3: A z-test for a proportion

H0: the true satisfaction rate is 0.95 (or higher). H1: it is lower than 0.95, so we test the left tail ('smaller'). We set prop_var=0.95 so the standard error uses the claimed value, as a test should.

In [3]:
from statsmodels.stats.proportion import proportions_ztest  # Load proportions_ztest, the z-test for a share of successes.

z_prop, p_prop = proportions_ztest(count=184, nobs=200, value=0.95,  # Run the test: 184 satisfied out of 200, claimed rate 0.95, alternative 'smaller' (left tail, H1: rate below 0.95).
                                   alternative='smaller', prop_var=0.95)  # prop_var=0.95 makes the standard error use the claimed rate, because a test assumes H0 is true.

print(f'sample proportion = {184 / 200:.2f}')  # Print the sample proportion, 184 / 200, to see how far it is from the claim.
print(f'z = {z_prop:.3f}')  # Print the z statistic. Negative means the sample is below the claim.
print(f'left-tailed p = {p_prop:.4f}')  # Print the left-tailed p-value.
print('reject H0 at 0.05?', p_prop <= 0.05)  # Compare p with alpha = 0.05 and print True or False for rejecting H0.

sample proportion = 0.92
z = -1.947
left-tailed p = 0.0258
reject H0 at 0.05? True


**What this means.** The sample proportion is 0.92, three points below the claim. z = -1.947 and the left-tailed p is 0.0258. If the true rate were 95%, a survey this low would happen about 2.6% of the time. That is below 0.05, so we reject H0: there is evidence the rate is below 95%. The test condition holds: 200 x 0.05 = 10.

## Step 4: Why the standard error matters

If we leave out prop_var, statsmodels uses the observed 0.92 in the standard error. This is the classic mistake from the lesson, and it changes z and p.

In [4]:
z_wrong, p_wrong = proportions_ztest(count=184, nobs=200, value=0.95,  # Run the same test but leave out prop_var, so statsmodels uses the observed proportion 0.92 in the standard error.
                                     alternative='smaller')  # The same call continues with alternative='smaller'; only the standard error differs from the previous step.

print(f'with the observed proportion: z = {z_wrong:.3f}, p = {p_wrong:.4f}')  # Print the result that wrongly uses the observed proportion.
print(f'with the claimed 0.95:         z = {z_prop:.3f}, p = {p_prop:.4f}')  # Print the correct result, using the claimed 0.95, side by side for comparison.

with the observed proportion: z = -1.564, p = 0.0589
with the claimed 0.95:         z = -1.947, p = 0.0258


**What this means.** Using the observed 0.92 in the standard error gives z = -1.564 and p = 0.0589, which would NOT be significant. Using the claimed 0.95, as a test should, gives z = -1.947 and p = 0.0258, which is. The decision flips on a detail. The test must ask 'what if H0 were true?', so the standard error uses p0.

## So, what is the answer?

Both claims fail at the 5% level, but only just. The bolt mean of 10.14 mm is significantly different from 10 mm (p = 0.0477), and the satisfaction rate of 92% is significantly below 95% (p = 0.0258). Caution: with p-values this close to 0.05, and an interval that barely excludes 10, treat these as warnings to check the line and collect more data, not as firm proof.

## Now you try

A hospital says that at most 10% of its patients are readmitted within 30 days. An auditor reviews 150 patient records and finds that 21 were readmitted. Is there evidence that the readmission rate is HIGHER than 10%? Use a one-sample z-test for a proportion, with the claimed rate in the standard error.

**Data:** readmitted, patients, p0 = 21, 150, 0.10   # readmitted patients, patients audited, claimed rate

1. Find the sample proportion and check n x p0 and n x (1 - p0) are at least 10.
2. Run the right-tailed z-test (alternative='larger') with prop_var=p0 and report z and p.
3. Run the two-sided version and compare the p-values.
4. Decide at alpha = 0.05 and say what the result does not show.

*Hint:* Use proportions_ztest(count, nobs, value=p0, alternative='larger', prop_var=p0). The two-sided p is double the one-sided p for a symmetric statistic.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

readmitted, patients, p0 = 21, 150, 0.10
print(f'sample proportion = {readmitted / patients:.3f}')
print('conditions:', patients * p0, patients * (1 - p0))
z, p = proportions_ztest(readmitted, patients, value=p0, alternative='larger', prop_var=p0)
print(f'z = {z:.3f}, right-tailed p = {p:.4f}')
z2, p2 = proportions_ztest(readmitted, patients, value=p0, alternative='two-sided', prop_var=p0)
print(f'two-sided p = {p2:.4f}')
print('reject H0 at 0.05?', p <= 0.05)

sample proportion = 0.140
conditions: 15.0 135.0
z = 1.633, right-tailed p = 0.0512
two-sided p = 0.1025
reject H0 at 0.05? False


**Answer.** The sample proportion is 0.140 and the conditions hold (15 and 135, both at least 10). z = 1.633 and the right-tailed p is 0.0512, just above 0.05, so we fail to reject H0; the two-sided p is 0.1025. The rate looks higher than 10%, but the evidence is borderline, not conclusive. It does not prove the rate is 10%. More records could settle it.

## The same question in R, SPSS and Stata

**R**

```r
# z-test for a mean (sigma known), by hand: base R has no z.test  # A comment (R ignores it). Base R has no z-test for a mean, so we compute it by hand.
z <- (10.14 - 10) / (0.5 / sqrt(50))  # Compute z = (10.14 - 10) / (0.5 / sqrt(50)): the gap in standard errors.
z  # Typing z alone prints its value.
2 * pnorm(-abs(z))  # Two-tailed p-value: twice the area below -|z| on the normal curve.

# z-test for a proportion, no continuity correction  # A comment: the next test is for a proportion.
res <- prop.test(184, 200, p = 0.95, alternative = "less", correct = FALSE)  # prop.test with p = 0.95 and alternative = 'less' is the left-tailed proportion test. correct = FALSE turns off the continuity correction so it matches the z-test.
res  # Print the whole result: X-squared, df, p-value and the sample estimate.
-sqrt(res$statistic)   # the z value (z squared is X-squared)  # prop.test reports X-squared, which is z squared; the negative square root gives z (negative because we are below the claim).
```

How to read it: First z prints 1.9799 and the two-sided p 0.04771, below 0.05 (reject H0 for the bolts). Then prop.test shows X-squared = 3.7895 and p-value = 0.02579, the left-tailed p for the satisfaction claim. The last line gives z = -1.9467.

**SPSS** (Transform > Compute Variable (SPSS has no z-test menu; use the normal-curve functions))

```
DATA LIST FREE / xbar mu0 sigma n x m p0.
BEGIN DATA
10.14 10 0.5 50 184 200 0.95
END DATA.
COMPUTE z_mean = (xbar - mu0) / (sigma / SQRT(n)).
COMPUTE p_mean = 2 * (1 - CDF.NORMAL(ABS(z_mean), 0, 1)).
COMPUTE z_prop = (x / m - p0) / SQRT(p0 * (1 - p0) / m).
COMPUTE p_prop = CDF.NORMAL(z_prop, 0, 1).
LIST z_mean p_mean z_prop p_prop.
```

How to read it: LIST shows z_mean 1.98 and p_mean 0.05 (display rounds 0.0477), then z_prop -1.95 and p_prop 0.03 (0.0258). Widen the columns or double-click the table to see more decimals. Both p-values are below 0.05, so both nulls are rejected.

**Stata**

```
display (10.14 - 10) / (0.5 / sqrt(50))
display 2 * (1 - normal(1.9799))
prtesti 200 184 0.95, count
```

How to read it: The first line shows 1.9799 and the second .04771. In the prtesti output, find z = -1.9467 and the left-tailed line Pr(Z < z) = 0.0258. Both are below 0.05, so both nulls are rejected.